# K-Nearest Neighbors (KNN) — Improved Model

This notebook improves upon the KNN baseline by:
- Tuning the number of neighbors (k) using cross-validation
- Comparing `weights='uniform'` vs `weights='distance'`
- Reporting the best configuration with full evaluation metrics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Load PCA-reduced data from the pre-processing folder
X_train = pd.read_csv('../pre-processing/X_train_pca.csv')
X_test  = pd.read_csv('../pre-processing/X_test_pca.csv')
y_train = pd.read_csv('../pre-processing/y_train.csv').values.ravel()
y_test  = pd.read_csv('../pre-processing/y_test.csv').values.ravel()

print('Training Features Shape:', X_train.shape)
print('Testing Features Shape: ', X_test.shape)
print('Class distribution (train):', pd.Series(y_train).value_counts().to_dict())

### Step 1: Tune K with Cross-Validation

We test `k = [3, 5, 7, 9, 11, 15, 21]` with both `weights='uniform'` and `weights='distance'`.
A 5-fold stratified cross-validation on the training set selects the best configuration
without ever touching the test set.

In [ ]:
k_values   = [3, 5, 7, 9, 11, 15, 21]
weight_types = ['uniform', 'distance']
cv = StratifiedKFold(n_splits=5, shuffle=False)

cv_results = {w: [] for w in weight_types}

for w in weight_types:
    print(f'\nweights = {w!r}')
    for k in k_values:
        knn   = KNeighborsClassifier(n_neighbors=k, weights=w)
        scores = cross_val_score(knn, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
        mean_acc = scores.mean()
        cv_results[w].append(mean_acc)
        print(f'  k={k:>2}: CV accuracy = {mean_acc:.4f} ± {scores.std():.4f}')

In [ ]:
# Plot k vs CV accuracy for both weight types
plt.figure(figsize=(9, 5))
for w in weight_types:
    plt.plot(k_values, cv_results[w], marker='o', label=f'weights={w!r}')

plt.xlabel('Number of Neighbors (k)')
plt.ylabel('CV Accuracy')
plt.title('KNN: Cross-Validation Accuracy vs. K')
plt.xticks(k_values)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('knn_k_tuning.png', dpi=150)
plt.show()
print('Plot saved as knn_k_tuning.png')

In [ ]:
# Find the best (k, weight) combination
best_acc   = -1
best_k     = None
best_weight = None

for w in weight_types:
    for i, k in enumerate(k_values):
        if cv_results[w][i] > best_acc:
            best_acc    = cv_results[w][i]
            best_k      = k
            best_weight = w

print(f'Best configuration: k={best_k}, weights={best_weight!r}')
print(f'Best CV accuracy:   {best_acc:.4f}')

### Step 2: Train the Best KNN Model and Evaluate on the Test Set

In [ ]:
best_knn = KNeighborsClassifier(n_neighbors=best_k, weights=best_weight)
best_knn.fit(X_train, y_train)
y_pred = best_knn.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
print(f'Best KNN — k={best_k}, weights={best_weight!r}')
print(f'Test Accuracy: {test_acc * 100:.2f}%')
print()
print('Classification Report:')
print('0 = Draw, 1 = Win, 2 = Loss')
print()
print(classification_report(y_test, y_pred, target_names=['Draw (0)', 'Win (1)', 'Loss (2)']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
labels = ['Draw (0)', 'Win (1)', 'Loss (2)']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Draw (0)', 'Win (1)', 'Loss (2)'], yticklabels=['Draw (0)', 'Win (1)', 'Loss (2)'])
plt.ylabel('Actual Outcome')
plt.xlabel('Predicted Outcome')
plt.title(f'KNN Confusion Matrix (k={best_k}, weights={best_weight!r})')
plt.tight_layout()
plt.savefig('knn_confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved as knn_confusion_matrix.png')

### Step 3: Compare Baseline vs Best KNN

In [ ]:
baseline_knn = KNeighborsClassifier(n_neighbors=5, weights='uniform')
baseline_knn.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, baseline_knn.predict(X_test))

print(f'Baseline KNN (k=5, uniform): {baseline_acc * 100:.2f}%')
print(f'Best KNN    (k={best_k}, {best_weight}): {test_acc * 100:.2f}%')
print(f'Improvement: +{(test_acc - baseline_acc) * 100:.2f}%')

plt.figure(figsize=(6, 5))
bars = plt.bar(
    [f'Baseline\n(k=5, uniform)', f'Best KNN\n(k={best_k}, {best_weight})'],
    [baseline_acc * 100, test_acc * 100],
    color=['steelblue', 'darkorange']
)
plt.ylim(0, 70)
plt.ylabel('Test Accuracy (%)')
plt.title('KNN: Baseline vs. Tuned Model')
for bar, val in zip(bars, [baseline_acc * 100, test_acc * 100]):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f'{val:.2f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('knn_baseline_vs_best.png', dpi=150)
plt.show()
print('Comparison chart saved as knn_baseline_vs_best.png')

### Summary

**Key results stored for comparison with SVM:**

In [ ]:
# Save best KNN result for cross-model comparison
import json
knn_result = {
    'model': 'KNN',
    'best_params': {'k': best_k, 'weights': best_weight},
    'test_accuracy': round(test_acc, 4),
    'baseline_accuracy': round(baseline_acc, 4)
}
with open('knn_best_result.json', 'w') as f:
    json.dump(knn_result, f, indent=2)
print(json.dumps(knn_result, indent=2))